# Homologação oficial — HTML/CSS/JavaScript via Spark e BBMagic

Suíte complementar ao `teste_html.ipynb`. Execute com a sessão BBMagic/Spark já conectada.

Este notebook:

- usa exclusivamente conteúdo sintético;
- não consulta DB2, Hive, HDFS, arquivos ou dados de cliente;
- prova a integridade de `get_from_spark` por tamanho e SHA-256;
- testa dashboard autocontido, CSS responsivo, JavaScript, eventos, SVG e Canvas;
- testa caracteres especiais e neutralização de HTML injetado;
- testa dois dashboards simultâneos;
- testa Plotly offline apenas quando a biblioteca já estiver disponível;
- diagnostica `send_to_spark` sem torná-lo requisito do dashboard;
- limpa os payloads grandes ao final.

Não execute células fora de ordem. A homologação técnica não substitui a conferência visual indicada no dashboard nativo.


In [ ]:
import gc
import hashlib
import html
import json
import platform
import re
import sys
import time
import traceback

from IPython import get_ipython
from IPython.display import HTML, Javascript, display

HML_RESULTADOS = {}

def hml_registrar(nome, status, detalhe=''):
    HML_RESULTADOS[nome] = {'status': status, 'detalhe': str(detalhe)}

spark_bbmagic_local = globals().get('spark')
if spark_bbmagic_local is None:
    raise RuntimeError('Sessão BBMagic local ausente. Inicialize a sessão antes desta suíte.')

get_from_spark_disponivel = callable(getattr(spark_bbmagic_local, 'get_from_spark', None))
send_to_spark_disponivel = callable(getattr(spark_bbmagic_local, 'send_to_spark', None))
if not get_from_spark_disponivel:
    raise RuntimeError('spark.get_from_spark não está disponível no objeto BBMagic local.')

ipython_atual = get_ipython()
cell_magics = set(ipython_atual.magics_manager.magics.get('cell', {}).keys()) if ipython_atual else set()

hml_registrar('KERNEL_LOCAL', 'OK', f'Python {sys.version.split()[0]} · {platform.platform()}')
hml_registrar('MAGIC_SPARK', 'OK' if 'spark' in cell_magics else 'FALHA', '%%spark registrada')
hml_registrar('GET_FROM_SPARK_API', 'OK', type(spark_bbmagic_local).__name__)
hml_registrar('SEND_TO_SPARK_API', 'DISPONIVEL' if send_to_spark_disponivel else 'INDISPONIVEL')

print('[HML][LOCAL] Python:', sys.version.replace('\n', ' '))
print('[HML][LOCAL] Objeto BBMagic:', type(spark_bbmagic_local))
print('[HML][LOCAL] get_from_spark:', get_from_spark_disponivel)
print('[HML][LOCAL] send_to_spark:', send_to_spark_disponivel)


## 1. Integridade e tamanho de `get_from_spark`

O Spark remoto cria cinco strings HTML autocontidas. O kernel local transfere cada variável separadamente, mede o tempo e compara tamanho e SHA-256. Os payloads grandes não são renderizados.


In [ ]:
%%spark

import hashlib
import json
import platform
import sys
import time

tamanhos_hml = {
    '1K': 1 * 1024,
    '100K': 100 * 1024,
    '500K': 500 * 1024,
    '1M': 1 * 1024 * 1024,
    '2M': 2 * 1024 * 1024,
}

def criar_html_tamanho_exato(tamanho_bytes, rotulo):
    prefixo = (
        '<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">'
        '<title>Homologação ' + rotulo + '</title></head><body>'
        '<div>áéíóú çãõ — Cliente sintético &amp; seguro</div><!--'
    )
    sufixo = '--></body></html>'
    bytes_fixos = len((prefixo + sufixo).encode('utf-8'))
    if bytes_fixos > tamanho_bytes:
        raise ValueError(f'Tamanho solicitado insuficiente: {tamanho_bytes}')
    return prefixo + ('X' * (tamanho_bytes - bytes_fixos)) + sufixo

metadados_payload_hml = {
    'runtime': {
        'python': sys.version.split()[0],
        'spark': spark.version,
        'master': spark.sparkContext.master,
        'java': spark.sparkContext._jvm.java.lang.System.getProperty('java.version'),
        'driver': platform.platform(),
    },
    'payloads': {},
}

for rotulo, tamanho in tamanhos_hml.items():
    inicio = time.perf_counter()
    conteudo = criar_html_tamanho_exato(tamanho, rotulo)
    nome_variavel = 'html_hml_' + rotulo.lower()
    globals()[nome_variavel] = conteudo
    metadados_payload_hml['payloads'][rotulo] = {
        'variavel': nome_variavel,
        'bytes': len(conteudo.encode('utf-8')),
        'sha256': hashlib.sha256(conteudo.encode('utf-8')).hexdigest(),
        'tempo_geracao_seg': time.perf_counter() - inicio,
    }

print('HML_PAYLOAD_REMOTO_BEGIN')
print(json.dumps(metadados_payload_hml, ensure_ascii=False, sort_keys=True))
print('HML_PAYLOAD_REMOTO_END')


In [ ]:
inicio_meta = time.perf_counter()
metadados_payload_local = spark_bbmagic_local.get_from_spark('metadados_payload_hml')
tempo_meta = time.perf_counter() - inicio_meta

resultados_transferencia_hml = []
for rotulo, esperado in metadados_payload_local['payloads'].items():
    inicio = time.perf_counter()
    try:
        conteudo_local = spark_bbmagic_local.get_from_spark(esperado['variavel'])
        tempo_transferencia = time.perf_counter() - inicio
        bytes_local = len(conteudo_local.encode('utf-8')) if isinstance(conteudo_local, str) else -1
        sha_local = hashlib.sha256(conteudo_local.encode('utf-8')).hexdigest() if isinstance(conteudo_local, str) else None
        unicode_ok = isinstance(conteudo_local, str) and 'áéíóú çãõ —' in conteudo_local
        aprovado = (
            isinstance(conteudo_local, str)
            and bytes_local == esperado['bytes']
            and sha_local == esperado['sha256']
            and unicode_ok
        )
        resultados_transferencia_hml.append({
            'rotulo': rotulo,
            'status': 'OK' if aprovado else 'FALHA',
            'bytes': bytes_local,
            'sha256_igual': sha_local == esperado['sha256'],
            'unicode_ok': unicode_ok,
            'tempo_transferencia_seg': tempo_transferencia,
        })
        del conteudo_local
    except Exception as exc:
        resultados_transferencia_hml.append({
            'rotulo': rotulo,
            'status': 'FALHA',
            'erro': f'{type(exc).__name__}: {exc}',
            'tempo_transferencia_seg': time.perf_counter() - inicio,
        })
    gc.collect()

transferencias_ok = all(item['status'] == 'OK' for item in resultados_transferencia_hml)
hml_registrar('GET_FROM_SPARK_INTEGRIDADE_ATE_2M', 'OK' if transferencias_ok else 'FALHA', resultados_transferencia_hml)
hml_registrar('GET_FROM_SPARK_METADADOS', 'OK', f'{tempo_meta:.6f}s')

print('HML_PAYLOAD_LOCAL_BEGIN')
print(json.dumps(resultados_transferencia_hml, ensure_ascii=False, indent=2))
print('HML_PAYLOAD_LOCAL_END')
if not transferencias_ok:
    print('[HML][ATENÇÃO] Um tamanho falhou; a suíte continuará para identificar as demais capacidades.')


## 2. Dashboard nativo autocontido

O Spark monta todo o HTML/CSS/JavaScript em uma string. O dashboard inclui valores nulos, zero, negativo, texto longo, caracteres especiais e uma tentativa sintética de injeção. Também cria dois dashboards com IDs independentes.


In [ ]:
%%spark

import hashlib
import html as html_lib
import json
import re

dados_hml = {
    'cliente': 'CLIENTE SINTÉTICO — ação & educação',
    'entrada': 8050.00,
    'saida': 4860.70,
    'saldo': 3189.30,
    'zero': 0,
    'negativo': -125.45,
    'nulo': None,
    'texto_longo': 'Descrição sintética muito longa para comprovar quebra de linha responsiva sem consultar qualquer fonte real.',
    'texto_hostil': '<script>window.RFH_XSS_EXECUTOU=true</script> "aspas" & conteúdo',
}

def gerar_dashboard_hml(root_id, titulo):
    esc = html_lib.escape
    payload_json = json.dumps(dados_hml, ensure_ascii=False).replace('</', '<\\/')
    template = r'''<section id="__ROOT__" class="rfh-root" data-homologacao="PENDENTE">
<style>
.rfh-root,.rfh-root *{box-sizing:border-box}.rfh-root{--ink:#172033;--muted:#657084;--blue:#2563eb;--cyan:#0891b2;--green:#047857;font-family:Inter,system-ui,sans-serif;color:var(--ink);background:linear-gradient(135deg,#f8fbff,#eef5ff);border:1px solid #dbe5f2;border-radius:18px;padding:22px;margin:12px 0}.rfh-head{display:flex;justify-content:space-between;gap:14px;align-items:flex-start}.rfh-title{margin:0;font-size:24px}.rfh-sub{color:var(--muted);margin:5px 0 0}.rfh-status{padding:7px 10px;border-radius:999px;background:#fff7ed;color:#9a3412;font-weight:800;font-size:12px}.rfh-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin-top:16px}.rfh-card{background:#fff;border:1px solid #e5eaf1;border-radius:13px;padding:14px;overflow-wrap:anywhere}.rfh-label{display:block;color:var(--muted);font-size:11px;text-transform:uppercase;font-weight:800}.rfh-value{display:block;font-size:22px;font-weight:850;margin-top:6px}.rfh-charts{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-top:10px}.rfh-bars{display:grid;gap:9px;margin-top:12px}.rfh-bar-row{display:grid;grid-template-columns:92px 1fr 62px;gap:8px;align-items:center;font-size:12px}.rfh-track{height:10px;background:#e9eef5;border-radius:999px;overflow:hidden}.rfh-fill{height:100%;background:linear-gradient(90deg,var(--blue),var(--cyan));border-radius:999px}.rfh-actions{margin-top:12px;display:flex;gap:10px;align-items:center;flex-wrap:wrap}.rfh-button{border:0;border-radius:9px;padding:9px 12px;background:#1d4ed8;color:white;font-weight:750;cursor:pointer}.rfh-report{white-space:pre-wrap;font:12px ui-monospace,monospace;background:#0f172a;color:#dbeafe;border-radius:10px;padding:11px;margin-top:12px}.rfh-hostile{margin-top:10px;padding:9px;background:#fff;border:1px dashed #94a3b8;border-radius:9px;font-size:12px}@media(max-width:850px){.rfh-grid{grid-template-columns:repeat(2,1fr)}.rfh-charts{grid-template-columns:1fr}}@media(max-width:500px){.rfh-root{padding:14px}.rfh-grid{grid-template-columns:1fr}}
</style>
<div class="rfh-head"><div><h2 class="rfh-title">__TITLE__</h2><p class="rfh-sub">__CLIENT__</p></div><span class="rfh-status" data-role="status">HOMOLOGAÇÃO VISUAL: PENDENTE</span></div>
<div class="rfh-grid">
 <article class="rfh-card"><span class="rfh-label">Entradas</span><strong class="rfh-value" data-money="8050">—</strong></article>
 <article class="rfh-card"><span class="rfh-label">Saídas</span><strong class="rfh-value" data-money="4860.70">—</strong></article>
 <article class="rfh-card"><span class="rfh-label">Zero / negativo</span><strong class="rfh-value">0 · −125,45</strong></article>
 <article class="rfh-card"><span class="rfh-label">Nulo</span><strong class="rfh-value">Não informado</strong></article>
</div>
<div class="rfh-charts">
 <article class="rfh-card"><strong>Barras CSS</strong><div class="rfh-bars"><div class="rfh-bar-row"><span>Moradia</span><div class="rfh-track"><div class="rfh-fill" style="width:92%"></div></div><span>92%</span></div><div class="rfh-bar-row"><span>Alimentação</span><div class="rfh-track"><div class="rfh-fill" style="width:58%"></div></div><span>58%</span></div><div class="rfh-bar-row"><span>Saúde</span><div class="rfh-track"><div class="rfh-fill" style="width:31%"></div></div><span>31%</span></div></div></article>
 <article class="rfh-card"><strong>SVG + Canvas</strong><div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;align-items:center"><svg data-role="svg" viewBox="0 0 180 100" style="width:100%" role="img" aria-label="Linha sintética"><polyline points="5,85 35,62 65,72 95,34 125,48 175,14" fill="none" stroke="#2563eb" stroke-width="6" stroke-linecap="round" stroke-linejoin="round"/></svg><canvas data-role="canvas" width="180" height="100" style="width:100%;background:#f8fafc;border-radius:8px"></canvas></div></article>
</div>
<div class="rfh-hostile"><b>Escape:</b> <span data-role="hostile">__HOSTILE__</span><br><b>Texto longo:</b> __LONG__</div>
<div class="rfh-actions"><button type="button" class="rfh-button" data-role="button">Testar evento</button><span data-role="clicks">Cliques: 0</span><span data-role="intl"></span></div>
<pre class="rfh-report" data-role="report">Testes do navegador ainda não executados.</pre>
<script type="application/json" data-role="payload">__PAYLOAD__</script>
<script>(function(){
 var root=document.getElementById('__ROOT__'); if(!root){return;} var checks=[];
 function check(name,ok){checks.push(name+'='+((ok)?'OK':'FALHA'));return ok;}
 var gridOk=getComputedStyle(root.querySelector('.rfh-grid')).display==='grid'; check('CSS_GRID',gridOk);
 var svgOk=!!root.querySelector('svg[data-role="svg"] polyline'); check('SVG',svgOk);
 var canvas=root.querySelector('canvas[data-role="canvas"]'),ctx=canvas&&canvas.getContext&&canvas.getContext('2d'),canvasOk=!!ctx;
 if(ctx){[38,62,47,82].forEach(function(v,i){ctx.fillStyle=['#bfdbfe','#93c5fd','#60a5fa','#2563eb'][i];ctx.fillRect(18+i*38,92-v,24,v);});} check('CANVAS',canvasOk);
 var fmt=new Intl.NumberFormat('pt-BR',{style:'currency',currency:'BRL'}); root.querySelectorAll('[data-money]').forEach(function(el){el.textContent=fmt.format(Number(el.dataset.money));});
 var intlOk=root.querySelector('[data-money]').textContent.indexOf('R$')>=0; root.querySelector('[data-role="intl"]').textContent='Intl: '+fmt.format(3189.30); check('INTL_PT_BR',intlOk);
 var payloadOk=false; try{payloadOk=JSON.parse(root.querySelector('[data-role="payload"]').textContent).cliente.indexOf('SINTÉTICO')>=0;}catch(e){} check('JSON',payloadOk);
 var xssOk=window.RFH_XSS_EXECUTOU!==true&&root.querySelector('[data-role="hostile"]').textContent.indexOf('<script>')>=0; check('ESCAPE_XSS',xssOk);
 var btn=root.querySelector('[data-role="button"]'),clicks=root.querySelector('[data-role="clicks"]'); if(!btn.dataset.bound){btn.dataset.bound='1';btn.dataset.count='0';btn.addEventListener('click',function(){btn.dataset.count=String(Number(btn.dataset.count)+1);clicks.textContent='Cliques: '+btn.dataset.count+' · evento OK';});} btn.click(); check('EVENTO_UNICO',btn.dataset.count==='1');
 var ids=Array.from(root.querySelectorAll('[id]')).map(function(el){return el.id;}),uniqueIds=(new Set(ids)).size===ids.length; check('IDS_INTERNOS',uniqueIds);
 var allOk=checks.every(function(item){return item.slice(-2)==='OK';}); root.dataset.homologacao=allOk?'OK':'FALHA'; var status=root.querySelector('[data-role="status"]');status.textContent='HOMOLOGAÇÃO VISUAL: '+(allOk?'OK':'FALHA');status.style.background=allOk?'#dcfce7':'#fee2e2';status.style.color=allOk?'#166534':'#991b1b';root.querySelector('[data-role="report"]').textContent=checks.join('\n');
})();</script></section>'''
    return (
        template
        .replace('__ROOT__', esc(root_id, quote=True))
        .replace('__TITLE__', esc(titulo))
        .replace('__CLIENT__', esc(dados_hml['cliente']))
        .replace('__HOSTILE__', esc(dados_hml['texto_hostil']))
        .replace('__LONG__', esc(dados_hml['texto_longo']))
        .replace('__PAYLOAD__', payload_json)
    )

html_dashboard_homologacao = (
    gerar_dashboard_hml('rfh-dashboard-a', 'Dashboard sintético A')
    + gerar_dashboard_hml('rfh-dashboard-b', 'Dashboard sintético B')
)
metadados_dashboard_hml = {
    'bytes': len(html_dashboard_homologacao.encode('utf-8')),
    'sha256': hashlib.sha256(html_dashboard_homologacao.encode('utf-8')).hexdigest(),
    'script_src_externo': bool(re.search(r'<script[^>]+src\s*=', html_dashboard_homologacao, flags=re.I)),
    'roots': ['rfh-dashboard-a', 'rfh-dashboard-b'],
}
print('HML_DASHBOARD_REMOTO_BEGIN')
print(json.dumps(metadados_dashboard_hml, ensure_ascii=False, sort_keys=True))
print('HML_DASHBOARD_REMOTO_END')


In [ ]:
inicio_dashboard = time.perf_counter()
html_dashboard_local = spark_bbmagic_local.get_from_spark('html_dashboard_homologacao')
metadados_dashboard_local = spark_bbmagic_local.get_from_spark('metadados_dashboard_hml')
tempo_dashboard = time.perf_counter() - inicio_dashboard

sha_dashboard_local = hashlib.sha256(html_dashboard_local.encode('utf-8')).hexdigest()
integridade_dashboard = (
    isinstance(html_dashboard_local, str)
    and len(html_dashboard_local.encode('utf-8')) == metadados_dashboard_local['bytes']
    and sha_dashboard_local == metadados_dashboard_local['sha256']
)
sem_dependencia_externa = not bool(re.search(r'<script[^>]+src\s*=|<link[^>]+href\s*=', html_dashboard_local, flags=re.I))
escape_estatico_ok = '&lt;script&gt;window.RFH_XSS_EXECUTOU=true&lt;/script&gt;' in html_dashboard_local
dois_roots_ok = all(html_dashboard_local.count(f'id="{root}"') == 1 for root in metadados_dashboard_local['roots'])

hml_registrar('DASHBOARD_SHA256', 'OK' if integridade_dashboard else 'FALHA', f'{metadados_dashboard_local["bytes"]} bytes · {tempo_dashboard:.6f}s')
hml_registrar('DASHBOARD_SEM_DEPENDENCIA_EXTERNA', 'OK' if sem_dependencia_externa else 'FALHA')
hml_registrar('DASHBOARD_ESCAPE_ESTATICO', 'OK' if escape_estatico_ok else 'FALHA')
hml_registrar('DASHBOARDS_SIMULTANEOS_IDS', 'OK' if dois_roots_ok else 'FALHA')

print('HML_DASHBOARD_LOCAL_BEGIN')
print(json.dumps({
    'bytes': len(html_dashboard_local.encode('utf-8')),
    'sha256_igual': integridade_dashboard,
    'sem_dependencia_externa': sem_dependencia_externa,
    'escape_estatico_ok': escape_estatico_ok,
    'dois_roots_ok': dois_roots_ok,
    'tempo_transferencia_seg': tempo_dashboard,
}, ensure_ascii=False, indent=2))
print('HML_DASHBOARD_LOCAL_END')
display(HTML(html_dashboard_local))


### Conferência visual obrigatória

Nos dois dashboards, confirme:

- selo `HOMOLOGAÇÃO VISUAL: OK`;
- relatório com todos os itens `=OK`;
- barras CSS, linha SVG e barras Canvas visíveis;
- valores `R$ 8.050,00`, `R$ 4.860,70` e `Intl: R$ 3.189,30`;
- tentativa `<script>...</script>` exibida como texto;
- botão começa em `Cliques: 1 · evento OK` e incrementa uma unidade por clique.

Reexecute uma vez a célula local anterior. O output deve ser substituído e o comportamento deve continuar idêntico, sem clique duplicado.


## 3. Plotly offline — capacidade opcional

O teste não instala bibliotecas. Se Plotly já estiver no Spark, gera um gráfico com JavaScript incorporado (`include_plotlyjs=True`), transfere o HTML e mede seu tamanho. A indisponibilidade não bloqueia a homologação nativa.


In [ ]:
%%spark

import hashlib
import json
import re

metadados_plotly_hml = {'status': 'INDISPONIVEL'}
html_plotly_hml = None
try:
    import plotly
    import plotly.graph_objects as go
    figura_hml = go.Figure(data=[
        go.Bar(x=['Moradia', 'Alimentação', 'Mobilidade', 'Saúde'], y=[2450, 1380.5, 620, 410.2], marker_color=['#2563eb','#0891b2','#059669','#ea580c'])
    ])
    figura_hml.update_layout(title='Gráfico sintético offline', template='plotly_white', height=420)
    html_plotly_hml = figura_hml.to_html(full_html=False, include_plotlyjs=True)
    metadados_plotly_hml = {
        'status': 'OK',
        'versao': plotly.__version__,
        'bytes': len(html_plotly_hml.encode('utf-8')),
        'sha256': hashlib.sha256(html_plotly_hml.encode('utf-8')).hexdigest(),
        'script_src_externo': bool(re.search(r'<script[^>]+src\s*=', html_plotly_hml, flags=re.I)),
    }
except Exception as exc:
    metadados_plotly_hml = {'status': 'INDISPONIVEL', 'detalhe': f'{type(exc).__name__}: {exc}'}

print('HML_PLOTLY_REMOTO_BEGIN')
print(json.dumps(metadados_plotly_hml, ensure_ascii=False, sort_keys=True))
print('HML_PLOTLY_REMOTO_END')


In [ ]:
metadados_plotly_local = spark_bbmagic_local.get_from_spark('metadados_plotly_hml')
if metadados_plotly_local['status'] == 'OK':
    inicio_plotly = time.perf_counter()
    html_plotly_local = spark_bbmagic_local.get_from_spark('html_plotly_hml')
    tempo_plotly = time.perf_counter() - inicio_plotly
    sha_plotly_ok = hashlib.sha256(html_plotly_local.encode('utf-8')).hexdigest() == metadados_plotly_local['sha256']
    offline_plotly = not bool(re.search(r'<script[^>]+src\s*=', html_plotly_local, flags=re.I))
    status_plotly = 'OK' if sha_plotly_ok and offline_plotly else 'FALHA'
    hml_registrar('PLOTLY_OFFLINE_OPCIONAL', status_plotly, {
        'versao': metadados_plotly_local['versao'],
        'bytes': metadados_plotly_local['bytes'],
        'sha256_igual': sha_plotly_ok,
        'sem_script_externo': offline_plotly,
        'tempo_transferencia_seg': tempo_plotly,
    })
    print('[HML][PLOTLY] Renderizando payload offline:', metadados_plotly_local['bytes'], 'bytes')
    display(HTML(html_plotly_local))
else:
    hml_registrar('PLOTLY_OFFLINE_OPCIONAL', 'INDISPONIVEL', metadados_plotly_local.get('detalhe', 'Biblioteca ausente.'))
    print('[HML][PLOTLY] INDISPONIVEL — não bloqueante:', metadados_plotly_local.get('detalhe'))


## 4. Diagnóstico opcional de `send_to_spark`

O dashboard não depende deste caminho, mas a documentação oficial pode registrar a capacidade bidirecional. A célula local envia um dicionário sintético; a célula remota valida sua presença sem consultar fontes.


In [ ]:
config_hml_local = {
    'origem': 'kernel-local',
    'mensagem': 'áéíóú çãõ — sintético',
    'versao': 1,
}
status_send_local = 'INDISPONIVEL'
detalhe_send_local = ''
if send_to_spark_disponivel:
    try:
        spark_bbmagic_local.send_to_spark(config_hml_local)
        status_send_local = 'ENVIADO'
    except Exception as exc:
        status_send_local = 'FALHA'
        detalhe_send_local = f'{type(exc).__name__}: {exc}'
print('[HML][SEND_TO_SPARK]', status_send_local, detalhe_send_local)


In [ ]:
%%spark

import hashlib
import json

resultado_send_to_spark_hml = {'status': 'INDISPONIVEL'}
try:
    recebido = config_hml_local
    aprovado = (
        isinstance(recebido, dict)
        and recebido.get('origem') == 'kernel-local'
        and recebido.get('mensagem') == 'áéíóú çãõ — sintético'
        and recebido.get('versao') == 1
    )
    resultado_send_to_spark_hml = {'status': 'OK' if aprovado else 'FALHA', 'recebido': recebido}
except Exception as exc:
    resultado_send_to_spark_hml = {'status': 'INDISPONIVEL', 'detalhe': f'{type(exc).__name__}: {exc}'}

print('HML_SEND_TO_SPARK_BEGIN')
print(json.dumps(resultado_send_to_spark_hml, ensure_ascii=False, sort_keys=True))
print('HML_SEND_TO_SPARK_END')


In [ ]:
try:
    resultado_send_local = spark_bbmagic_local.get_from_spark('resultado_send_to_spark_hml')
    hml_registrar('SEND_TO_SPARK_OPCIONAL', resultado_send_local['status'], resultado_send_local)
except Exception as exc:
    hml_registrar('SEND_TO_SPARK_OPCIONAL', 'INDISPONIVEL', f'{type(exc).__name__}: {exc}')
print('[HML][SEND_TO_SPARK][RESULTADO]', HML_RESULTADOS['SEND_TO_SPARK_OPCIONAL'])


## 5. Resultado técnico

Plotly e `send_to_spark` são informativos. A homologação obrigatória exige runtime local, `%%spark`, integridade até 2 MB, integridade do dashboard, ausência de dependência externa, escape estático e IDs independentes.


In [ ]:
requisitos_obrigatorios = [
    'KERNEL_LOCAL',
    'MAGIC_SPARK',
    'GET_FROM_SPARK_API',
    'GET_FROM_SPARK_INTEGRIDADE_ATE_2M',
    'GET_FROM_SPARK_METADADOS',
    'DASHBOARD_SHA256',
    'DASHBOARD_SEM_DEPENDENCIA_EXTERNA',
    'DASHBOARD_ESCAPE_ESTATICO',
    'DASHBOARDS_SIMULTANEOS_IDS',
]
faltantes = [nome for nome in requisitos_obrigatorios if nome not in HML_RESULTADOS]
falhas = [nome for nome in requisitos_obrigatorios if HML_RESULTADOS.get(nome, {}).get('status') != 'OK']
homologacao_tecnica = 'S' if not faltantes and not falhas else 'N'
HML_RESULTADOS['HOMOLOGACAO_TECNICA_HTML_SPARK'] = {
    'status': homologacao_tecnica,
    'detalhe': {'faltantes': faltantes, 'falhas': falhas},
}

linhas = []
for nome, resultado in HML_RESULTADOS.items():
    status = resultado['status']
    ok = status in ('OK', 'S', 'DISPONIVEL', 'ENVIADO')
    cor = '#166534' if ok else ('#92400e' if status == 'INDISPONIVEL' else '#991b1b')
    fundo = '#dcfce7' if ok else ('#fef3c7' if status == 'INDISPONIVEL' else '#fee2e2')
    linhas.append('<tr><td style="padding:8px;border-bottom:1px solid #e5e7eb"><b>'+html.escape(nome)+'</b></td><td style="padding:8px;border-bottom:1px solid #e5e7eb"><span style="padding:4px 7px;border-radius:999px;color:'+cor+';background:'+fundo+';font-weight:800">'+html.escape(status)+'</span></td><td style="padding:8px;border-bottom:1px solid #e5e7eb;color:#475569">'+html.escape(str(resultado.get('detalhe','')))+'</td></tr>')

display(HTML('<div style="font-family:system-ui;border:1px solid #cbd5e1;border-radius:15px;padding:17px"><h3 style="margin-top:0">Homologação técnica HTML/Spark: '+homologacao_tecnica+'</h3><table style="width:100%;border-collapse:collapse">'+''.join(linhas)+'</table><p style="padding:10px;background:#eff6ff;border-radius:9px"><b>Pendente para documentação oficial:</b> confirmar visualmente os dois dashboards nativos e, quando disponível, o gráfico Plotly.</p></div>'))

print('HOMOLOGACAO_TECNICA_HTML_SPARK_BEGIN')
print(json.dumps(HML_RESULTADOS, ensure_ascii=False, indent=2))
print('HOMOLOGACAO_TECNICA_HTML_SPARK_END')


## 6. Limpeza remota

Execute somente depois de copiar os resultados. Remove da memória do driver as strings usadas na homologação; não remove arquivos nem dados.


In [ ]:
%%spark

nomes_limpeza_hml = [
    'html_hml_1k', 'html_hml_100k', 'html_hml_500k', 'html_hml_1m', 'html_hml_2m',
    'metadados_payload_hml', 'html_dashboard_homologacao', 'metadados_dashboard_hml',
    'html_plotly_hml', 'metadados_plotly_hml', 'resultado_send_to_spark_hml',
]
removidos_hml = []
for nome in nomes_limpeza_hml:
    if nome in globals():
        globals().pop(nome, None)
        removidos_hml.append(nome)
print('[HML][LIMPEZA] Variáveis remotas removidas:', removidos_hml)
